# Machine Learning — Complete End-to-End Example
## Dataset: Salaries

This notebook walks through the **entire ML workflow** from raw data to trained model.
Every step is commented in detail so you can use this as a reference.

**Goal:** Predict a person's salary based on physical and demographic features.

**Type:** Regression (salary is a continuous number)

---
**Steps:**
1. Load & explore the data
2. Clean & preprocess
3. Visualize
4. Split into train/test
5. Train model
6. Evaluate
7. Improve

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler

%matplotlib inline

## 2. Load the Data

We load the CSV file into a pandas DataFrame.
A DataFrame is essentially a table — rows are people, columns are their properties.

In [ ]:
# Load the dataset
df = pd.read_csv('salaries.csv')

# Always start by looking at the first few rows to understand the structure
df.head()

## 3. Explore the Data (EDA — Exploratory Data Analysis)

Before touching the data, we need to understand what we're working with.

In [ ]:
# Shape tells us how many rows (people) and columns (features) we have
print(f"Shape: {df.shape}")
print(f"→ {df.shape[0]} people, {df.shape[1]} features\n")

# dtypes tells us the data type of each column
# 'object' means text/string — these can't go directly into an ML model
print("Data types:")
print(df.dtypes)

In [ ]:
# describe() gives us statistics for all numeric columns:
# count, mean, std (spread), min, 25%/50%/75% quartiles, max
# Use this to spot outliers (is the max value realistic?)
df.describe()

In [ ]:
# Check for missing values — missing data will crash most ML algorithms
print("Missing values per column:")
print(df.isnull().sum())

print(f"\nTotal missing: {df.isnull().sum().sum()}")

In [ ]:
# Check for duplicate rows — they can bias the model
print(f"Duplicate rows: {df.duplicated().sum()}")

In [ ]:
# Check categorical columns — what categories exist?
print("Experience values:", df['Experience'].unique())
print("Gender values:", df['Gender'].unique())
print("Daltonic values:", df['Daltonic'].unique())

## 4. Visualize

Visualization helps us understand relationships between features and spot potential issues.

In [ ]:
# Pair plot: shows the relationship between every pair of numeric features
# Diagonal: distribution of each feature
# Off-diagonal: scatter plot between two features
# → Look for: linear trends, clusters, outliers
sns.pairplot(df.select_dtypes(include='number'))
plt.suptitle('Pair Plot — All Numeric Features', y=1.02)
plt.show()

In [ ]:
# Correlation heatmap: shows correlation coefficients between all numeric features
# Value range: -1 (perfect negative) to +1 (perfect positive)
# 0 = no linear relationship
# Dark red = strong positive correlation
# Dark blue = strong negative correlation
plt.figure(figsize=(10, 8))
sns.heatmap(
    df.select_dtypes(include='number').corr(),
    annot=True,        # show numbers in cells
    fmt='.2f',         # round to 2 decimal places
    cmap='coolwarm'    # color scheme: blue=negative, red=positive
)
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
# Look at salary distribution by experience level
# This helps confirm whether Experience is a useful predictor for Salary
plt.figure(figsize=(8, 5))
sns.boxplot(x='Experience', y='Salary', data=df)
plt.title('Salary Distribution by Experience Level')
plt.show()

## 5. Preprocess the Data

Machine learning models require **all-numeric input**.
We need to:
1. Handle missing values in 'Daltonic'
2. Convert text categories to numbers (one-hot encoding)

In [ ]:
# Step 1: Fill missing values in 'Daltonic'
# The NaN values represent people without color vision deficiency
# We fill them with 'None' so they get their own category during encoding
df['Daltonic'] = df['Daltonic'].fillna('None')

print("Daltonic after filling NaN:")
print(df['Daltonic'].value_counts())

In [ ]:
# Step 2: One-hot encode all categorical columns
# pd.get_dummies creates a new binary column for each unique value
# Example: 'Experience' with values 'Junior'/'Senior'
# → becomes 'Experience_Junior' (0 or 1) and 'Experience_Senior' (0 or 1)
# This is necessary because a model can't understand 'Junior' as text
df = pd.get_dummies(df, columns=['Experience', 'Gender', 'Daltonic'])

# Convert True/False booleans to 1/0 integers
# get_dummies creates boolean columns by default
df = df.apply(lambda x: x.astype(int) if x.dtype == bool else x)

print("Columns after encoding:")
print(df.columns.tolist())

In [ ]:
# Verify everything looks clean
df.describe()

## 6. Split into Features (X) and Target (y)

- **X** = all features (everything the model uses as input)
- **y** = target (what we want to predict — Salary)

In [ ]:
# X contains all columns EXCEPT the target
# We drop 'Salary' because that's what we're trying to predict
X = df.drop(columns=['Salary'])

# y contains only the target column
y = df['Salary']

print(f"X shape: {X.shape}  → {X.shape[0]} samples, {X.shape[1]} features")
print(f"y shape: {y.shape}  → {y.shape[0]} target values")

## 7. Train/Test Split

We split the data so we can evaluate the model on data it has **never seen during training**.
- Training set (80%): the model learns from this
- Test set (20%): we use this to measure real-world performance

In [ ]:
# test_size=0.2 → 20% goes to test, 80% to training
# random_state=42 → fixed random seed so we always get the same split
#   (important for reproducibility — without this, results change every run)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(f"Training set:  {X_train.shape[0]} samples")
print(f"Test set:      {X_test.shape[0]} samples")

## 8. Train the Model

We use **Linear Regression** as our first model.

Linear Regression fits a line (or hyperplane) through the data:
`Salary = β₀ + β₁·Height + β₂·Weight + β₃·Experience_Senior + ...`

The model finds the best coefficients (β values) that minimize prediction error.

In [ ]:
# Initialize the model
# At this point, no learning has happened yet — we just created an empty model
model = LinearRegression()

# Train the model on the training data
# .fit() is where the actual learning happens
# The model finds the best coefficients for each feature
model.fit(X_train, y_train)

print("Model trained!")
print(f"Intercept (β₀): {model.intercept_:.2f}")
print("\nCoefficients (one per feature):")
for feature, coef in zip(X.columns, model.coef_):
    print(f"  {feature}: {coef:.2f}")

## 9. Evaluate the Model

We evaluate on both training and test data:
- **R² on train**: how well the model fit the training data
- **R² on test**: how well it generalizes to unseen data

If train score >> test score → **overfitting** (model memorized training data)

In [ ]:
# Generate predictions on both sets
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# R² Score: ranges from -∞ to 1.0
# 1.0 = perfect, 0.0 = predicts the mean, negative = worse than the mean
r2_train = r2_score(y_train, y_train_pred)
r2_test = r2_score(y_test, y_test_pred)

# MAE: Mean Absolute Error — average error in the same unit as Salary ($)
mae = mean_absolute_error(y_test, y_test_pred)

print(f"R² Train:  {r2_train:.4f}")
print(f"R² Test:   {r2_test:.4f}")
print(f"MAE Test:  ${mae:,.2f}")

gap = r2_train - r2_test
if gap > 0.1:
    print(f"\n⚠️  Gap of {gap:.3f} between train/test — possible overfitting")
else:
    print(f"\n✓  Small gap ({gap:.3f}) — model generalizes well")

In [ ]:
# Visualize: Actual vs. Predicted values
# A perfect model would have all points on the diagonal line
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_test_pred, alpha=0.6, color='steelblue', label='Predictions')

# Draw perfect prediction line
min_val = min(y_test.min(), y_test_pred.min())
max_val = max(y_test.max(), y_test_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', label='Perfect prediction')

plt.xlabel('Actual Salary')
plt.ylabel('Predicted Salary')
plt.title(f'Actual vs. Predicted Salary (R² = {r2_test:.3f})')
plt.legend()
plt.tight_layout()
plt.show()

## 10. Improve — Try More Training Data (90/10 Split)

In [ ]:
# Try giving the model more training data (90% instead of 80%)
# More training data often = better model
X_train90, X_test90, y_train90, y_test90 = train_test_split(
    X, y, test_size=0.1, random_state=42
)

model90 = LinearRegression()
model90.fit(X_train90, y_train90)

y_pred90 = model90.predict(X_test90)
r2_90 = r2_score(y_test90, y_pred90)

print(f"R² with 80% training data: {r2_test:.4f}")
print(f"R² with 90% training data: {r2_90:.4f}")

if r2_90 > r2_test:
    print("→ More training data improved the model")
else:
    print("→ More training data did not improve the model")

## 11. Feature Importance

Which features actually matter for predicting salary?
We can look at the absolute value of the coefficients.
Larger absolute coefficient = bigger influence on the prediction.

In [ ]:
# Create a DataFrame of feature importances
importance = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_,
    'Abs_Coefficient': abs(model.coef_)
}).sort_values('Abs_Coefficient', ascending=False)

print("Feature importance (sorted by absolute coefficient):")
print(importance.to_string(index=False))

In [ ]:
# Visualize feature importance
plt.figure(figsize=(10, 6))
colors = ['green' if c > 0 else 'red' for c in importance['Coefficient']]
plt.barh(importance['Feature'], importance['Coefficient'], color=colors)
plt.axvline(x=0, color='black', linewidth=0.8)
plt.xlabel('Coefficient Value')
plt.title('Feature Coefficients\n(green = positive effect, red = negative effect)')
plt.tight_layout()
plt.show()

## Summary — Part 1 (Regression)

**What we did:**
1. Loaded and explored the salaries dataset (200 rows, 11 columns)
2. Found and handled missing values in 'Daltonic'
3. One-hot encoded categorical columns (Experience, Gender, Daltonic)
4. Split data 80/20 into train/test
5. Trained a Linear Regression model
6. Evaluated with R² and MAE
7. Analyzed which features had the most influence

**Key insight:** Physical features (Height, Weight, BMI) had near-zero correlation with Salary.

## Final Summary

### Part 1 — Regression (predict Salary)
- **Linear Regression** — fits a hyperplane, evaluated with R² and MAE
- Physical features barely correlated with salary

### Part 2 — Classification (predict Junior/Senior)
- Reformulated same dataset as a binary classification problem

| Algorithm | Key strength | Key weakness | Needs scaling |
|-----------|-------------|-------------|---------------|
| Logistic Regression | Fast, probabilistic output, strong baseline | Linear boundary only | Yes |
| KNN | No training time, flexible boundary | Slow predictions, needs scaling | **Yes** |
| Decision Tree | Fully interpretable (read the tree!), captures non-linear patterns | Overfits without max_depth | No |

### Key takeaways
1. Always start simple — Logistic Regression is a great baseline
2. KNN **always** needs feature scaling — results without it are misleading
3. Decision Trees overfit without `max_depth` — always check train vs test per depth
4. No single best algorithm — compare and pick based on your data and goals

In [ ]:
# All three confusion matrices side by side
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (preds, title) in zip(axes, [
    (y_pred_lr,  f'Logistic Regression\n({acc_lr:.3f})'),
    (y_pred_knn, f'KNN K={best_k}\n({acc_knn:.3f})'),
    (y_pred_dt,  f'Decision Tree depth=4\n({acc_dt:.3f})')
]):
    ConfusionMatrixDisplay(confusion_matrix(y_test_c, preds), display_labels=le.classes_).plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(title)
plt.tight_layout()
plt.show()

In [ ]:
# Side-by-side comparison
comparison = pd.DataFrame({
    'Model': ['Logistic Regression', f'KNN (K={best_k})', 'Decision Tree (depth=4)'],
    'Accuracy': [acc_lr, acc_knn, acc_dt],
    'Needs Scaling': ['Yes', 'Yes', 'No'],
    'Interpretability': ['Coefficients', 'Low', 'High (tree viz)']
}).sort_values('Accuracy', ascending=False).reset_index(drop=True)

print(comparison.to_string(index=False))

# Bar chart
plt.figure(figsize=(8, 4))
bars = plt.barh(comparison['Model'], comparison['Accuracy'], color=['#4f8ef7', '#34a853', '#ea4335'])
for bar, val in zip(bars, comparison['Accuracy']):
    plt.text(bar.get_width() - 0.01, bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', ha='right', color='white', fontweight='bold')
plt.xlim(0, 1.05)
plt.xlabel('Test Accuracy')
plt.title('Classification Model Comparison')
plt.tight_layout()
plt.show()

## 16. Model Comparison

In [ ]:
# Feature importance — which features drove the splits most?
feat_imp = pd.Series(dt_final.feature_importances_, index=X_clf.columns)
feat_imp[feat_imp > 0].sort_values().plot(kind='barh', figsize=(9, 5), color='steelblue')
plt.title('Decision Tree — Feature Importance')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

In [ ]:
# Train final tree with max_depth=4 (adjust based on table above)
dt_final = DecisionTreeClassifier(max_depth=4, random_state=42, criterion='gini')
dt_final.fit(X_train_c, y_train_c)

y_pred_dt = dt_final.predict(X_test_c)
acc_dt = accuracy_score(y_test_c, y_pred_dt)

print(f"Decision Tree Accuracy: {acc_dt:.4f}\n")
print(classification_report(y_test_c, y_pred_dt, target_names=le.classes_))

# Visualize the tree — biggest advantage: you can READ the decision rules
plt.figure(figsize=(20, 8))
plot_tree(dt_final, feature_names=X_clf.columns, class_names=le.classes_,
          filled=True, rounded=True, fontsize=9)
plt.title('Decision Tree (max_depth=4)')
plt.show()

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

# Check overfitting at different depths — without max_depth the tree memorizes training data
print(f"{'Depth':>6} | {'Train':>7} | {'Test':>7}")
print("-" * 28)
for depth in range(1, 11):
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt.fit(X_train_c, y_train_c)   # no scaling needed for Decision Trees
    print(f"{depth:>6} | {dt.score(X_train_c, y_train_c):>7.3f} | {dt.score(X_test_c, y_test_c):>7.3f}")

## 15. Decision Tree

A Decision Tree learns a series of **yes/no questions** about features to split data into groups.
Splits are chosen to minimize **Gini impurity** (how mixed a node is — 0 = pure).

**No feature scaling needed** — only thresholds matter, not distances.

In [ ]:
# Plot train vs test accuracy — small K = overfitting, large K = underfitting
plt.figure(figsize=(10, 5))
plt.plot(results_df['K'], results_df['Train'], marker='o', label='Train Accuracy')
plt.plot(results_df['K'], results_df['Test'],  marker='o', label='Test Accuracy')
plt.axvline(x=best_row['K'], color='gray', linestyle='--', alpha=0.7, label=f'Best K={int(best_row["K"])}')
plt.xlabel('K')
plt.ylabel('Accuracy')
plt.title('KNN: Train vs. Test Accuracy for K=1..20')
plt.legend()
plt.tight_layout()
plt.show()

# Final model with best K
best_k = int(best_row['K'])
knn_final = KNeighborsClassifier(n_neighbors=best_k)
knn_final.fit(X_train_s, y_train_c)
y_pred_knn = knn_final.predict(X_test_s)
acc_knn = accuracy_score(y_test_c, y_pred_knn)

print(f"KNN (K={best_k}) Accuracy: {acc_knn:.4f}\n")
print(classification_report(y_test_c, y_pred_knn, target_names=le.classes_))

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# Step 1: Scale — fit on train ONLY, then transform both
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train_c)
X_test_s  = scaler.transform(X_test_c)

# Step 2: Find best K
results = []
for k in range(1, 21):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_s, y_train_c)
    results.append({'K': k, 'Train': knn.score(X_train_s, y_train_c), 'Test': knn.score(X_test_s, y_test_c)})

results_df = pd.DataFrame(results)
best_row = results_df.loc[results_df['Test'].idxmax()]
print(results_df.to_string(index=False))
print(f"\nBest K: {int(best_row['K'])} → Test Accuracy: {best_row['Test']:.4f}")

## 14. K-Nearest Neighbors (KNN)

KNN classifies a new point by finding its **K closest neighbors** and taking a majority vote.

**Rule:** Always scale features before KNN — distances are meaningless without it.

In [ ]:
from sklearn.linear_model import LogisticRegression

# max_iter=1000 because the default 100 often doesn't converge
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_c, y_train_c)

y_pred_lr = lr.predict(X_test_c)
acc_lr = accuracy_score(y_test_c, y_pred_lr)

print(f"Logistic Regression Accuracy: {acc_lr:.4f}\n")
print(classification_report(y_test_c, y_pred_lr, target_names=le.classes_))

# Confusion matrix
ConfusionMatrixDisplay(confusion_matrix(y_test_c, y_pred_lr), display_labels=le.classes_).plot(cmap='Blues')
plt.title('Logistic Regression — Confusion Matrix')
plt.show()

# Probabilities — each row = [P(Junior), P(Senior)]
proba = lr.predict_proba(X_test_c)
print("Sample probabilities (first 5):")
print(pd.DataFrame(proba, columns=['P(Junior)', 'P(Senior)']).head())

## 13. Logistic Regression

Despite the name, this is a **classification** algorithm.
It passes a linear combination through the sigmoid function: `σ(z) = 1 / (1 + e^(-z))` to output a probability between 0 and 1.
If probability ≥ 0.5 → Senior, else → Junior.

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# Reload raw data — Experience is now the TARGET, not a feature
df_raw = pd.read_csv('salaries.csv')
df_raw['Daltonic'] = df_raw['Daltonic'].fillna('None')

# Encode Gender and Daltonic only (NOT Experience — that's our target)
df_clf = pd.get_dummies(df_raw, columns=['Gender', 'Daltonic'])
df_clf = df_clf.apply(lambda x: x.astype(int) if x.dtype == bool else x)

# Encode target: Junior=0, Senior=1
le = LabelEncoder()
y_clf = le.fit_transform(df_clf['Experience'])

print("Classes:", le.classes_)
print("Distribution:", pd.Series(y_clf).value_counts().to_dict())

# Features = everything except Experience
X_clf = df_clf.drop(columns=['Experience'])

# Train/test split
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42
)
print(f"\nTrain: {len(X_train_c)} | Test: {len(X_test_c)}")

## 12. Prepare Data for Classification

---
# Part 2: Classification

Now we switch from **regression** to **classification**.

**New goal:** Predict whether a person is **Junior or Senior** based on their features (including salary).

**Type:** Binary Classification (2 classes: Junior / Senior)

We compare three algorithms: Logistic Regression, KNN, Decision Tree.